# guard
> Check text before it leaves the machine.

In [ ]:
#| default_exp guard
#| export
from ramabana.agent import mask_secrets, masking_guard, pii_report
from ramabana.core import AgentError, PII_MODES, PII_OFF
from ramabana.setup import setting
from shalya.core import ReadCheck, Refusal

In [ ]:
#| export
class OutgoingRefused(AgentError):
    "Text was withheld before it left the machine."


def _policy(mode=None, ner=None):
    mode = setting('cli.pii', PII_OFF) if mode is None else mode
    if mode not in PII_MODES: raise ValueError(f'unknown pii mode: {mode!r}; choose one of {", ".join(PII_MODES)}')
    return mode, bool(setting('cli.pii_ner', False) if ner is None else ner)


def _refused(why): raise OutgoingRefused(why)


def _need_detector():
    "Refuse the text when no PII detector is installed."
    if pii_report is None: _refused('text was withheld because PII detection is unavailable')


def _has_pii(text, ner):
    _need_detector()
    return pii_report(text, mx=max(len(text), 1), ner=ner).has_pii


def _vault_text(vault, refs, text, model, mode, ner):
    if vault is None:
        if refs: _refused('vault references were withheld because no vault can check them')
        return text
    try: checked = vault.send_check(refs=refs, text=text, model=model, policy=mode, ner=ner)
    except Exception as e: raise OutgoingRefused(f'vault content was withheld because its privacy check failed: {type(e).__name__}') from e
    if not checked.ok: _refused('vault content was withheld by its privacy policy')
    if any(row.how == 'masked' and row.doc_id for row in checked.rows):
        _refused('vault references were withheld because marked personal content cannot be safely masked here')
    return next((row.text for row in checked.rows if row.get('text') is not None), text)


def outgoing(text, model, mode=None, ner=None, vault=None, refs=None):
    "Return text that may be sent to model, or raise `OutgoingRefused`."
    mode, ner = _policy(mode, ner)
    text = _vault_text(vault, refs, mask_secrets(text), model, mode, ner)
    local = model.local
    if not local and mode != PII_OFF: _need_detector()
    check = ReadCheck('outgoing text', 'outgoing', 'url', lambda: text, model=getattr(model, 'name', ''), local=local)
    guarded = masking_guard(lambda: (mode, ner))(check)
    if isinstance(guarded, Refusal): _refused('text was withheld by the personal-data policy')
    return text if guarded is None else guarded


def query(text, ner=None):
    "Return an external query without secrets or personal data."
    ner = bool(setting('cli.pii_ner', False) if ner is None else ner)
    masked = mask_secrets(text)
    if masked != text: _refused('query was withheld because it contains a secret')
    if _has_pii(text, ner): _refused('query was withheld because it contains personal data')
    return text